# TrendLens — 02 · Embeddings (Phase 2)

**Goal:** load the 5,000-image sample, compute CLIP ViT-B/32 embeddings for every image, and save the (5000, 512) embedding matrix alongside aligned metadata.

> **DATA INTEGRITY NOTICE:** only the image files and the path index are real. Engagement metadata (likes, comments, timestamps, tags, geo) is **synthetic demo data** and is flagged as such throughout. Nothing derived from it is a research finding.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "config.py").exists():
    for p in Path.cwd().parents:
        if (p / "config.py").exists():
            REPO = p
            break
sys.path.insert(0, str(REPO))

import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.embeddings import load_clip, embed_images, l2_normalize

## 1 · Load the 5,000-image sample

In [ ]:
sample_path = config.SAMPLE_METADATA_PATH
if not sample_path.exists():
    raise FileNotFoundError(
        f"Sample metadata not found at {sample_path}.\n"
        "Run notebook 01_data_exploration.ipynb first to generate the 5,000-image sample."
    )

df = pd.read_parquet(sample_path)
print(f"Sample: {len(df)} images")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 2 · Load the CLIP model

We use **ViT-B/32** from OpenAI — 512-dimensional embeddings, fast inference.

In [ ]:
model, processor, device = load_clip()
print(f"CLIP loaded on {device}")

## 3 · Compute CLIP embeddings

This processes every image through the CLIP vision encoder and L2-normalizes the output.

In [ ]:
%%time
emb, meta = embed_images(df, model, processor, device)
print(f"Embeddings: {emb.shape}")
print(f"Metadata rows: {len(meta)}")

## 4 · Embedding statistics

In [ ]:
print(f"Shape:  {emb.shape}")
print(f"Dtype:  {emb.dtype}")
print(f"Mean:   {emb.mean():.6f}")
print(f"Std:    {emb.std():.6f}")
print(f"Min:    {emb.min():.6f}")
print(f"Max:    {emb.max():.6f}")
print(f"L2 norm (first 5): {np.linalg.norm(emb[:5], axis=1)}")

## 5 · Pairwise distance distribution

Understanding how spread out the embeddings are helps tune downstream clustering parameters.

In [ ]:
from sklearn.metrics import pairwise_distances

# Sample for speed if dataset is large
sample_n = min(500, len(emb))
idx = np.random.choice(len(emb), sample_n, replace=False)
dists = pairwise_distances(emb[idx])
dflat = dists[dists > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(dflat, bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].set_title("Pairwise L2 distance distribution")
axes[0].set_xlabel("Distance")
axes[0].set_ylabel("Count")

percentiles = [10, 25, 50, 75, 90]
vals = [np.percentile(dflat, p) for p in percentiles]
axes[1].barh([str(p) for p in percentiles], vals, color="steelblue")
axes[1].set_title("Distance percentiles")
axes[1].set_xlabel("Distance")

fig.tight_layout()
plt.show()

print(f"Mean distance: {dflat.mean():.4f}")
print(f"Std distance:  {dflat.std():.4f}")
for p, v in zip(percentiles, vals):
    print(f"  {p}th percentile: {v:.4f}")

## 6 · 2D visualization (t-SNE)

Quick sanity check — do visually similar images cluster together?

In [ ]:
from sklearn.manifold import TSNE

sample_n = min(1000, len(emb))
idx = np.random.choice(len(emb), sample_n, replace=False)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb_2d = tsne.fit_transform(emb[idx])

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=8, alpha=0.6, c="steelblue")
ax.set_title("t-SNE of CLIP embeddings (sampled)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
fig.tight_layout()
plt.show()

## 7 · Verify nearest neighbors

For a few random images, show their 5 nearest neighbors by embedding distance.

In [ ]:
from PIL import Image

def show_neighbors(emb, meta, idx, k=5):
    """Show an image and its k nearest neighbors."""
    query_emb = emb[idx]
    dists = np.linalg.norm(emb - query_emb, axis=1)
    nearest = np.argsort(dists)[1 : k + 1]  # skip self
    
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3))
    
    # Query image
    path = Path(meta.iloc[idx]["image_path"])
    if path.exists():
        axes[0].imshow(Image.open(path))
    axes[0].set_title(f"Query (#{idx})", fontsize=9)
    axes[0].axis("off")
    
    for j, ni in enumerate(nearest, 1):
        path = Path(meta.iloc[ni]["image_path"])
        if path.exists():
            axes[j].imshow(Image.open(path))
        axes[j].set_title(f"Neighbor {j}\nd={dists[ni]:.3f}", fontsize=8)
        axes[j].axis("off")
    
    fig.tight_layout()
    plt.show()

# Show 3 random examples
for _ in range(3):
    show_neighbors(emb, meta, np.random.randint(len(emb)))

## 8 · Save embeddings

Save the embedding matrix and aligned metadata for downstream stages (clustering, retrieval).

In [ ]:
np.save(config.EMBEDDINGS_PATH, emb)
meta.to_parquet(config.CLUSTER_METADATA_DIR / "embed_meta.parquet", index=False)
print(f"Saved embeddings: {config.EMBEDDINGS_PATH}")
print(f"Saved metadata:   {config.CLUSTER_METADATA_DIR / 'embed_meta.parquet'}")
print(f"Shape: {emb.shape}, dtype: {emb.dtype}")

## Summary

| Item | Value |
|------|-------|
| Model | CLIP ViT-B/32 (OpenAI) |
| Embedding dim | 512 |
| Normalization | L2 (unit vectors) |
| Images embedded | 5,000 |
| Output | `data/embeddings.npy` + `embed_meta.parquet` |

**Next:** `03_clustering.ipynb` — UMAP dimensionality reduction + HDBSCAN clustering.